<a href="https://colab.research.google.com/github/MMujtabaX/agentic-ai-journey/blob/main/01_llm_api_basics/llm_api_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · LLM API Basics: First Steps Toward Agentic AI

Calling an LLM from code, streaming responses, and **chaining calls** so the output of one becomes the input of the next: the core building block of every agent.

Uses **Groq** (free tier, very fast inference). The notebook picks the best available models automatically, since Groq retires model names over time.

> 🔑 Add `GROQ_API_KEY` to Colab Secrets (🔑 icon in the left sidebar). Get a free key at [console.groq.com](https://console.groq.com/keys).

In [ ]:
!pip install -q groq

In [ ]:
import os
from google.colab import userdata
from groq import Groq
from IPython.display import Markdown, display

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("✅ Groq client ready")

In [ ]:
# ─── Pick a model that your Groq account can access ──────────────────────────
# Groq retires models over time (e.g. llama-3.1-8b-instant was deprecated in 2026),
# so we list what's available and choose the first match from a preference list.
available = sorted(m.id for m in client.models.list().data)
print("Available models:", available)

def pick(preferences):
    for name in preferences:
        if name in available:
            return name
    chat_models = [m for m in available if not any(x in m for x in ("whisper", "tts", "guard", "embed"))]
    return chat_models[0]

FAST_MODEL = pick(["openai/gpt-oss-20b", "llama-3.1-8b-instant", "llama3-8b-8192"])
SMART_MODEL = pick(["llama-3.3-70b-versatile", "openai/gpt-oss-120b", "openai/gpt-oss-20b"])
print(f"\n✅ Fast model : {FAST_MODEL}\n✅ Smart model: {SMART_MODEL}")

## 1. A first call
Messages use the familiar `role` / `content` format shared by OpenAI-compatible APIs.

In [ ]:
messages = [{"role": "user", "content": "What is 2 + 2? Answer in one word."}]
response = client.chat.completions.create(model=FAST_MODEL, messages=messages)
print(response.choices[0].message.content)

## 2. Streaming a long response
`stream=True` returns tokens as they're generated, which is how chat UIs show text "typing" in real time.

> Note: parameters like `reasoning_effort` only work with reasoning models. Passing it to a non-reasoning model returns a `400 BadRequestError`, so it's omitted here.

In [ ]:
stream = client.chat.completions.create(
    model=FAST_MODEL,
    messages=[{"role": "user", "content": "Write a short article (about 250 words) on use cases of AI in healthcare."}],
    temperature=0.7,
    max_completion_tokens=600,
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="")

## 3. Chaining two calls
The model writes a hard question, then a second call answers it. Two LLM calls, wired together in code.

In [ ]:
MODEL = SMART_MODEL

def ask(prompt, model=MODEL):
    r = client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

question = ask("Propose a hard, challenging question to assess someone's IQ. Respond only with the question.")
print("❓", question)
display(Markdown(ask(question)))

## 4. Exercise: an agentic workflow in three calls

1. Ask the LLM for a **business area** ripe for an agentic AI solution
2. Ask for the biggest **pain point** in that area
3. Ask it to propose an **agentic AI solution** for that pain point

Each step's output feeds the next prompt. This is the simplest form of a multi-step agent pipeline.

In [ ]:
business_area = ask("Name ONE business area that is ripe for an agentic AI solution. "
                    "Reply with the area name and one sentence explaining why.")
display(Markdown(f"### 🏢 Business area\n{business_area}"))

pain_point = ask(f"In this business area: {business_area}\n\n"
                 "What is the single most painful, expensive problem professionals face? "
                 "Describe it in 3-4 sentences.")
display(Markdown(f"### 😣 Pain point\n{pain_point}"))

solution = ask(f"Pain point: {pain_point}\n\n"
               "Propose an agentic AI solution: which agents, which tools each agent uses, "
               "and how they hand work to each other. Keep it under 200 words.")
display(Markdown(f"### 🤖 Agentic solution\n{solution}"))